# 2.3 SQL + Python Integration> SQLAlchemy for relational DBs, DuckDB for analytical queries on local data.---

In [ ]:
import duckdb# DuckDB — in-process analytical database (like SQLite for analytics)# No server needed, reads Parquet/CSV directlycon = duckdb.connect()# Create sample datacon.execute("""    CREATE TABLE patients AS    SELECT * FROM (VALUES        ('P001', 45, 'diabetes', 6.5),        ('P002', 62, 'hypertension', 140.0),        ('P003', 38, 'diabetes', 7.1),        ('P004', 71, 'copd', NULL),        ('P005', 55, 'hypertension', 135.0)    ) AS t(patient_id, age, diagnosis, lab_value)""")# SQL queries return DataFramesresult = con.execute("""    SELECT diagnosis,            COUNT(*) as count,           AVG(age) as avg_age,           AVG(lab_value) as avg_lab    FROM patients    GROUP BY diagnosis    ORDER BY count DESC""").fetchdf()print(result)

In [ ]:
# 💡 DuckDB reads Pandas/Polars DataFrames directly!import pandas as pdencounters = pd.DataFrame({    "patient_id": ["P001", "P001", "P002", "P003"],    "encounter_date": pd.to_datetime(["2024-01-15", "2024-03-20", "2024-02-10", "2024-04-01"]),    "cost": [1200.5, 800.0, 2100.75, 500.0],})# Query the DataFrame with SQL — incredibly usefulresult = duckdb.sql("""    SELECT p.patient_id, p.diagnosis,            COUNT(e.encounter_date) as visits,           SUM(e.cost) as total_cost    FROM patients p    JOIN encounters e USING (patient_id)    GROUP BY p.patient_id, p.diagnosis""").fetchdf()print(result)

## SQLAlchemy — ORM & Connection Management☕ **Java parallel:** Like JPA/Hibernate but lighter. You can use raw SQL or the ORM.

In [ ]:
from sqlalchemy import create_engine, text# Create in-memory SQLite for demoengine = create_engine("sqlite:///:memory:")# Raw SQL execution (like JDBC)with engine.connect() as conn:    conn.execute(text("""        CREATE TABLE pipeline_runs (            id INTEGER PRIMARY KEY,            name TEXT,            status TEXT,            records_processed INTEGER        )    """))    conn.execute(text("""        INSERT INTO pipeline_runs VALUES        (1, 'etl_load', 'success', 50000),        (2, 'etl_transform', 'failed', 0),        (3, 'etl_validate', 'success', 49800)    """))    conn.commit()        # Pandas reads directly from SQLAlchemy    df = pd.read_sql("SELECT * FROM pipeline_runs", conn)    print(df)